In [1]:
# Install if needed:
# !pip install pandas scikit-learn matplotlib seaborn nltk

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

nltk.download("stopwords")

# 1. Load dataset
df = pd.read_csv("geo_subdivision_idn_adm3.csv")

# 2. Select text column
TEXT = "name"   # change if your column has another name
df[TEXT] = df[TEXT].fillna("").astype(str)

# 3. Clean text
def clean(x):
    x = x.lower()
    x = re.sub(r"[^a-z\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

df["text"] = df[TEXT].apply(clean)

# 4. Indonesian stopwords
stop = set(stopwords.words("indonesian"))
stop.update(["kabupaten", "kecamatan", "desa", "kelurahan", "kota"])

# 5. NLP: TF-IDF
tfidf = TfidfVectorizer(stop_words=list(stop), ngram_range=(1,2))
X = tfidf.fit_transform(df["text"])

# 6. Find best K
scores = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)

# 7. K-Means
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(X)

# 8. Show results
print(df[[TEXT, "cluster"]].head(20))
print("\nCluster counts:")
print(df["cluster"].value_counts().sort_index())

# 9. PCA visualization
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X.toarray())

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=coords[:,0],
    y=coords[:,1],
    hue=df["cluster"],
    palette="tab10"
)
plt.title("K-Means Clustering")
plt.show()

# 10. Save result
df.to_csv("geo_subdivision_idn_adm3_clustered.csv", index=False)
print("Saved successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


KeyError: 'name'